# Week 6 Listing Signal Extraction

This notebook reviews the listing-level signal output built from Week 3 entity extraction and Week 1 taxonomy patterns.


In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

---

## 1. Artifacts Loading

Load the full listing signal artifact. It contains only product-facing fields and represents the complete local `rets_property` table.


In [2]:
signal_path = Path("../data/processed/listing_signals.jsonl")

records = [json.loads(line) for line in signal_path.open()]
signals = pd.DataFrame(records)

signals.shape


(53122, 4)

In [3]:
signals.head(3)


,listing_id,text_signals,numeric_signals,keywords
0,1000291026,"{'amenities': [], 'condition': ['move in ready'], 'interior_features': [], 'exterior_features': ['fenced', 'solar'], 'location_features': ['view'], 'financing_terms': [], 'transaction_features': [], 'investment_features': [], 'rooms': [], 'property_type': [], 'parking': ['garage']}","{'price': 1499900, 'beds': 3, 'baths': 2, 'sqft': 1680, 'lot_size': 112, 'year_built': None, 'hoa_fee': None, 'stories': None}","[fenced, garage, move in ready, solar, view]"
1,1000295679,"{'amenities': ['pool'], 'condition': [], 'interior_features': [], 'exterior_features': [], 'location_features': ['beach'], 'financing_terms': [], 'transaction_features': [], 'investment_features': [], 'rooms': [], 'property_type': [], 'parking': []}","{'price': 2495000, 'beds': 2, 'baths': 2, 'sqft': 1588, 'lot_size': 0.25, 'year_built': None, 'hoa_fee': None, 'stories': 2}","[beach, pool]"
2,1000314732,"{'amenities': ['fitness center', 'lounge', 'pool', 'spa'], 'condition': ['updated'], 'interior_features': ['appliances', 'fireplace', 'high ceilings', 'main floor'], 'exterior_features': ['acre lot', 'landscaping', 'yard'], 'location_features': ['near dining', 'view'], 'financing_terms': [], 'transaction_features': [], 'investment_features': [], 'rooms': ['chef s kitchen', 'guest suite', 'primary suite'], 'property_type': [], 'parking': []}","{'price': 7295000, 'beds': 6, 'baths': 10, 'sqft': 7130, 'lot_size': None, 'year_built': None, 'hoa_fee': None, 'stories': None}","[acre lot, appliances, chef s kitchen, fireplace, fitness center, guest suite, high ceilings, landscaping, lounge, main floor, near dining, pool, primary suite, spa, updated, view, yard]"


---

## 2. Signal Coverage

Coverage shows which buckets are common enough to support filtering or ranking. Sparse buckets are still useful, but they should be treated carefully in evaluation.


In [4]:
numeric = pd.json_normalize(signals["numeric_signals"])
signal_buckets = list(records[0]["text_signals"])

def bucket_values(record, bucket):
    return record["text_signals"][bucket]


In [5]:
coverage = []
for bucket in signal_buckets:
    count = sum(bool(bucket_values(record, bucket)) for record in records)
    coverage.append(
        {
            "signal_type": "text",
            "signal": bucket,
            "listings_with_signal": count,
            "coverage": count / len(records),
        }
    )

for field in numeric:
    count = numeric[field].notna().sum()
    coverage.append(
        {
            "signal_type": "numeric",
            "signal": field,
            "listings_with_signal": count,
            "coverage": count / len(records),
        }
    )

pd.DataFrame(coverage).sort_values(["signal_type", "coverage"], ascending=[True, False])


,signal_type,signal,listings_with_signal,coverage
11,numeric,price,53122,1.000000
13,numeric,baths,53112,0.999812
14,numeric,sqft,53067,0.998965
12,numeric,beds,53024,0.998155
18,numeric,stories,11515,0.216765
15,numeric,lot_size,8390,0.157938
16,numeric,year_built,3156,0.059410
17,numeric,hoa_fee,1711,0.032209
19,numeric,lot_size.width_ft,5,0.000094
20,numeric,lot_size.depth_ft,5,0.000094


---

## 3. Common Signals

Review the most frequent values by bucket. This gives a quick read on what the extractor is actually surfacing from listing remarks.


In [6]:
def top_values(bucket, n=5):
    counter = Counter()
    for record in records:
        counter.update(bucket_values(record, bucket))
    return pd.DataFrame(counter.most_common(n), columns=[bucket, "count"])


In [7]:
top_values("amenities")


,amenities,count
0,pool,20625
1,spa,15509
2,fitness center,6162
3,clubhouse,4321
4,golf course,3029


In [8]:
top_values("condition")


,condition,count
0,updated,18257
1,remodeled,12301
2,move in ready,8390
3,well maintained,7450
4,freshly painted,6131


In [9]:
top_values("location_features")


,location_features,count
0,near shopping,14435
1,near dining,12071
2,view,11582
3,freeway access,9710
4,near schools,9029


---

## 4. Extraction Evaluation

This section measures whether the extracted output agrees with reviewed gold labels.


In [10]:
evaluation_path = Path("../data/processed/listing_signal_eval_results.json")
evaluation = json.loads(evaluation_path.read_text())


- **Structured field accuracy**: compares `price`, `beds`, `baths`, and `sqft` against the corresponding MLS values for each listing.
- **Free-text F1**: compares extracted `(bucket, canonical value)` pairs with reviewed remark labels, balancing correct matches against both missed and extra signals.
- **Keyword integrity**: checks that `keywords` is exactly the deduplicated, flattened set of string values in `text_signals`.


In [11]:
pd.DataFrame(
    [
        {"metric": "Structured field accuracy", "score": evaluation["structured_fields"]["accuracy"]},
        {"metric": "Free-text F1", "score": evaluation["free_text"]["f1"]},
        {"metric": "Keyword integrity", "score": evaluation["keyword_integrity"]["accuracy"]},
    ]
).style.format({"score": "{:.1%}"})


,metric,score
0,Structured field accuracy,100.0%
1,Free-text F1,79.9%
2,Keyword integrity,100.0%



Free-text matching uses exact `(bucket, canonical value)` pairs.


In [12]:
text_metrics = pd.DataFrame.from_dict(
    evaluation["free_text"]["per_bucket"], orient="index"
).rename_axis("bucket").reset_index()

text_metrics[["bucket", "support", "precision", "recall", "f1"]].sort_values("f1")


,bucket,support,precision,recall,f1
5,investment_features,36,0.818182,0.500000,0.620690
10,transaction_features,28,0.589744,0.821429,0.686567
6,location_features,361,0.813725,0.689751,0.746627
8,property_type,87,0.777778,0.724138,0.750000
9,rooms,165,0.728723,0.830303,0.776204
1,condition,205,0.806931,0.795122,0.800983
7,parking,112,0.836364,0.821429,0.828829
4,interior_features,382,0.848000,0.832461,0.840159
0,amenities,122,0.832000,0.852459,0.842105
2,exterior_features,189,0.820000,0.867725,0.843188
